# Upload and Deploy Notebook

This notebook:
1. logs in to Hugging Face
2. reloads base model + LoRA adapter
3. merges the adapter into the base model
4. pushes the merged model to the Hub
5. tests prompting from the uploaded model repo
  

In [ ]:
# !pip -q install -U "transformers>=4.46.0" "peft>=0.11.1" "accelerate>=0.30.1" \
#                    "huggingface_hub>=0.23.0" "safetensors>=0.4.3"

In [ ]:
!pip install -U transformers peft accelerate safetensors huggingface_hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login, HfApi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Login and set repo name

In [ ]:
# Login with a token stored in Colab Secrets as HF_TOKEN
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets. Add it in the Secrets pane and enable notebook access.")

login(HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_DIR = "/content/drive/MyDrive/Thesis/FineTunning/llama31_lora_adapter_lbl"
MERGED_DIR = "/content/drive/MyDrive/Thesis/FineTunning/llama31_merged_lbl"

HF_USERNAME = "usmanmanzoor"
REPO_NAME = "llama31-radiotherapy-classifier"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

print("Target repo:", REPO_ID)


Target repo: usmanmanzoor/llama31-radiotherapy-classifier


In [ ]:
# Load base model in fp16 and attach the LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("Base + adapter loaded.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Base + adapter loaded.


In [ ]:
# Merge adapter into the base model and save locally
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="5GB",
)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Merged model saved to: /content/drive/MyDrive/Thesis/FineTunning/llama31_merged_lbl


In [ ]:
# Create the Hub repo and push the merged model
api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)

merged_model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print("Uploaded to:", REPO_ID)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fdwmp2b/model.safetensors:   0%|          | 16.0MB / 16.1GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpal2wgu2r/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Uploaded to: usmanmanzoor/llama31-radiotherapy-classifier


## Prompt again from the uploaded Hub repo

In [ ]:
# Load back from the Hub repo to verify the upload
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

LABELS = ["Radiotherapy", "No radiation"]

def make_prompt_from_values(age, sex, marital_status, primary_site, grade, t_stage, n_stage, m_stage):
    return f"""Decide the radiotherapy outcome.

Age: {age}
Sex: {sex}
MaritalStatus: {marital_status}
PrimarySite: {primary_site}
Grade: {grade}
TNM: {t_stage} {n_stage} {m_stage}

Answer with exactly one of:
Radiotherapy
No radiation

Answer:
""".strip()

def build_label_token_ids(tok, labels):
    seqs = []
    for lab in labels:
        for variant in [lab, " " + lab]:
            ids = tok(variant, add_special_tokens=False)["input_ids"]
            if ids:
                seqs.append(ids)
    uniq, seen = [], set()
    for s in seqs:
        t = tuple(s)
        if t not in seen:
            uniq.append(s)
            seen.add(t)
    return uniq

label_token_seqs = build_label_token_ids(tokenizer, LABELS)

def make_prefix_allowed_fn(prompt_len):
    eos = tokenizer.eos_token_id

    def prefix_allowed_tokens_fn(batch_id, input_ids):
        seq = input_ids if input_ids.dim() == 1 else input_ids[batch_id]
        gen = seq[prompt_len:].tolist()

        if len(gen) == 0:
            return sorted({seq_ids[0] for seq_ids in label_token_seqs})

        allowed = set()
        complete = False

        for seq_ids in label_token_seqs:
            if gen == seq_ids:
                complete = True
                continue
            if len(gen) < len(seq_ids) and gen == seq_ids[:len(gen)]:
                allowed.add(seq_ids[len(gen)])

        if complete:
            return [eos] if eos is not None else []
        if allowed:
            return sorted(allowed)
        return [eos] if eos is not None else []

    return prefix_allowed_tokens_fn


def predict_case(age, sex, marital_status, primary_site, grade, t_stage, n_stage, m_stage):
    prompt = make_prompt_from_values(age, sex, marital_status, primary_site, grade, t_stage, n_stage, m_stage)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=6,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            prefix_allowed_tokens_fn=make_prefix_allowed_fn(prompt_len),
        )

    pred = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return pred if pred in LABELS else "No radiation"

prediction = predict_case(
    age=55,
    sex="Male",
    marital_status="Married",
    primary_site="Tongue",
    grade="Moderately differentiated",
    t_stage="T2",
    n_stage="N1",
    m_stage="M0",
)

print("Prediction:", prediction)

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prediction: Radiotherapy


## Next step

Create a Hugging Face Space and upload the `app.py` and `requirements.txt` files from the companion package.